<a href="https://colab.research.google.com/github/Erika4chaos/OPERADOR-4-OPERACOES/blob/main/TRABALHO_cOMPILADOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Disciplina: Linguagens Formais e Compiladores

Grupo:


*   Erika Oliveira Silva
*   Henrique Souza Uchida
*   Hian Araujo Damaceno






# Compilador Didático em Python

Esse notebook implementa um compilador do zero para uma linguagem bem simples. A ideia foi aplicar na prática as **5 fases clássicas** que todo compilador passa:

1. **Análise Léxica** — o `Lexer` lê o código e separa em pedaços
2. **Análise Sintática** — o `Parser` monta a estrutura do programa
3. **Análise Semântica** — o `SemanticAnalyzer` verifica se o código faz sentido
4. **Geração de Código** — o `CodeGenerator` transforma em instruções simples
5. **Execução** — a `VirtualMachine` roda essas instruções

A linguagem que criamos aceita: declaração de variáveis com `let`, reatribuição, `print(...)`, as 4 operações aritméticas, números inteiros e decimais, e comentários com `#`.

## Importações

Antes de começar, importamos três módulos padrão do Python:

- **`Enum` e `auto`**: usados para criar uma lista de constantes com nomes (os tipos de token). O `auto()` gera os valores automaticamente, sem precisar numerar na mão.
- **`dataclass`**: um atalho do Python que evita escrever `__init__` repetidamente em classes que só guardam dados.
- **`field`**: complemento do `dataclass`, usado para definir valores padrão mais complexos, como uma lista vazia.

In [36]:
from enum import Enum, auto
from dataclasses import dataclass, field

## Fase 1 — Análise Léxica (Lexer)

O Lexer é a primeira coisa que roda. Ele recebe o código-fonte inteiro como texto e quebra tudo em **tokens** — que são as menores unidades com significado no código.

Por exemplo, a linha `let x = 10 + 2` vira: `LET`, `IDENTIFIER(x)`, `ASSIGN(=)`, `INTEGER(10)`, `PLUS(+)`, `INTEGER(2)`.

### `TokenType`
Define todos os tipos de token que a linguagem conhece. É uma enumeração — basicamente uma lista de nomes fixos. Tem tipos para números (`INTEGER`, `FLOAT`), nomes de variáveis (`IDENTIFIER`), palavras-chave (`LET`, `PRINT`), operadores (`PLUS`, `MINUS`, `STAR`, `SLASH`), símbolos (`ASSIGN`, `LPAREN`, `RPAREN`, `SEMICOLON`) e marcadores especiais (`NEWLINE`, `EOF`).

### `Token`
É a estrutura que representa um token encontrado. Guarda: o tipo (`tipo`), o valor real (`valor`), a linha (`linha`) e a coluna (`coluna`) — as duas últimas servem para mostrar onde um erro aconteceu.

### `ErroLexico`
Exceção lançada quando o Lexer encontra um caractere que não faz parte da linguagem, como `@` ou `$`. A mensagem já mostra a linha e coluna exatas.

### Classe `Lexer`
O motor dessa fase. Os métodos principais são:

- `char_atual()` e `proximo_char()`: olham o caractere atual e o próximo sem avançar
- `avancar()`: avança um caractere, atualizando a posição de linha/coluna
- `pular_espacos_e_comentarios()`: pula espaços e comentários que começam com `#`
- `ler_numero()`: lê um número; se tiver `.`, vira `FLOAT`, senão `INTEGER`
- `ler_identificador()`: lê uma palavra; verifica se é `let`/`print` ou nome de variável
- `tokenizar()`: o método principal — percorre tudo e devolve a lista de tokens

Um detalhe importante: o `-` pode ser operador de subtração (`a - 5`) ou parte de um número negativo (`-5`). O `tokenizar()` resolve isso olhando o token anterior — se vier depois de número, variável ou `)`, é operador; senão, faz parte do número.

In [37]:
# tipos de token que a linguagem aceita
class TokenType(Enum):
    # literais
    INTEGER  = auto()
    FLOAT    = auto()

    # nome de variavel
    IDENTIFIER = auto()

    # palavras reservadas
    LET   = auto()
    PRINT = auto()

    # operadores
    PLUS  = auto()
    MINUS = auto()
    STAR  = auto()
    SLASH = auto()

    # outros simbolos
    ASSIGN    = auto()
    LPAREN    = auto()
    RPAREN    = auto()
    SEMICOLON = auto()
    NEWLINE   = auto()
    EOF       = auto()


@dataclass
class Token:
    tipo: TokenType
    valor: any
    linha: int
    coluna: int


class ErroLexico(Exception):
    def __init__(self, msg, linha, col):
        super().__init__(f'Erro Léxico (linha {linha}, col {col}): {msg}')
        self.linha = linha
        self.col   = col


class Lexer:
    def __init__(self, codigo):
        self.codigo = codigo
        self.pos    = 0
        self.linha  = 1
        self.col    = 1

    def char_atual(self):
        if self.pos >= len(self.codigo):
            return None
        return self.codigo[self.pos]

    def proximo_char(self):
        if self.pos + 1 >= len(self.codigo):
            return None
        return self.codigo[self.pos + 1]

    def avancar(self):
        ch = self.codigo[self.pos]
        self.pos += 1
        if ch == '\n':
            self.linha += 1
            self.col = 1
        else:
            self.col += 1
        return ch

    def pular_espacos_e_comentarios(self):
        while self.char_atual() in (' ', '\t', '\r', '#'):
            if self.char_atual() == '#':
                # ignora o resto da linha
                while self.char_atual() is not None and self.char_atual() != '\n':
                    self.avancar()
            else:
                self.avancar()

    def ler_numero(self):
        col_inicio = self.col
        numero = ''
        while self.char_atual() and self.char_atual().isdigit():
            numero += self.avancar()

        # se tiver ponto, é float
        if self.char_atual() == '.':
            numero += self.avancar()
            while self.char_atual() and self.char_atual().isdigit():
                numero += self.avancar()
            return Token(TokenType.FLOAT, float(numero), self.linha, col_inicio)

        return Token(TokenType.INTEGER, int(numero), self.linha, col_inicio)

    def ler_identificador(self):
        col_inicio = self.col
        palavra = ''
        while self.char_atual() and (self.char_atual().isalnum() or self.char_atual() == '_'):
            palavra += self.avancar()

        # verifica se é palavra reservada
        palavras_reservadas = {
            'let':   TokenType.LET,
            'print': TokenType.PRINT
        }
        tipo = palavras_reservadas.get(palavra, TokenType.IDENTIFIER)
        return Token(tipo, palavra, self.linha, col_inicio)

    def tokenizar(self):
        lista_tokens = []

        while True:
            self.pular_espacos_e_comentarios()

            if self.char_atual() is None:
                lista_tokens.append(Token(TokenType.EOF, 'EOF', self.linha, self.col))
                break

            ch  = self.char_atual()
            lin = self.linha
            col = self.col

            if ch == '\n':
                self.avancar()
                lista_tokens.append(Token(TokenType.NEWLINE, '\n', lin, col))

            elif ch.isdigit():
                lista_tokens.append(self.ler_numero())

            elif ch == '-' and self.proximo_char() and self.proximo_char().isdigit():
                # decide se é operador ou número negativo
                tipo_anterior = lista_tokens[-1].tipo if lista_tokens else None
                if tipo_anterior in (TokenType.INTEGER, TokenType.FLOAT,
                                     TokenType.IDENTIFIER, TokenType.RPAREN):
                    self.avancar()
                    lista_tokens.append(Token(TokenType.MINUS, '-', lin, col))
                else:
                    self.avancar()
                    tok = self.ler_numero()
                    tok.valor = -tok.valor
                    tok.coluna = col
                    lista_tokens.append(tok)

            elif ch.isalpha() or ch == '_':
                lista_tokens.append(self.ler_identificador())

            elif ch == '+':
                self.avancar()
                lista_tokens.append(Token(TokenType.PLUS, '+', lin, col))
            elif ch == '-':
                self.avancar()
                lista_tokens.append(Token(TokenType.MINUS, '-', lin, col))
            elif ch == '*':
                self.avancar()
                lista_tokens.append(Token(TokenType.STAR, '*', lin, col))
            elif ch == '/':
                self.avancar()
                lista_tokens.append(Token(TokenType.SLASH, '/', lin, col))
            elif ch == '=':
                self.avancar()
                lista_tokens.append(Token(TokenType.ASSIGN, '=', lin, col))
            elif ch == '(':
                self.avancar()
                lista_tokens.append(Token(TokenType.LPAREN, '(', lin, col))
            elif ch == ')':
                self.avancar()
                lista_tokens.append(Token(TokenType.RPAREN, ')', lin, col))
            elif ch == ';':
                self.avancar()
                lista_tokens.append(Token(TokenType.SEMICOLON, ';', lin, col))
            else:
                raise ErroLexico(f"caractere '{ch}' não reconhecido", lin, col)

        return lista_tokens

print('Lexer carregado')

Lexer carregado


## Nós da AST e Padrão Visitor

### O que é a AST?
Depois que o Lexer gera os tokens, o Parser os transforma numa árvore que representa a estrutura do programa. Essa árvore se chama **AST (Árvore Sintática Abstrata)**.

Ela é "abstrata" porque joga fora detalhes que não importam (como parênteses e ponto-e-vírgula) e mantém só a hierarquia das operações.

`let x = 2 + 3 * 4` por exemplo vira:

DeclaracaoLet: x

OperacaoBinaria: '+'

LiteralInt: 2

OperacaoBinaria: '*'

LiteralInt: 3

LiteralInt: 4


### Classes de nós
Cada tipo de construção da linguagem tem sua classe. `LiteralInt` e `LiteralFloat` são as folhas da árvore (valores puros). `OperacaoBinaria` representa qualquer operação com dois lados. `OperacaoUnaria` é a negação (`-x`). `DeclaracaoLet`, `Atribuicao` e `ComandoPrint` são as instruções. `Programa` é a raiz — contém a lista de tudo.

### Padrão Visitor
O Visitor é uma forma de percorrer a árvore sem misturar a lógica dentro de cada nó. O método `visitar(no)` detecta automaticamente o tipo do nó e chama o método certo — por exemplo, se receber uma `OperacaoBinaria`, chama `visitar_OperacaoBinaria()`.

Isso permite criar visitantes diferentes para propósitos diferentes: o `ImpressorAST` imprime a árvore, o `AnalisadorSemantico` verifica tipos, o `GeradorDeCodigo` gera bytecode — tudo sem alterar as classes dos nós.

### `ImpressorAST`
Um visitante simples que imprime a AST com indentação, mostrando a hierarquia. Aparece quando você usa `verbose=True`.

In [38]:
# classe base pra todos os nós da árvore
class No: pass

@dataclass
class LiteralInt(No):
    valor: int

@dataclass
class LiteralFloat(No):
    valor: float

@dataclass
class Variavel(No):
    nome:  str
    linha: int

@dataclass
class OperacaoBinaria(No):
    op:      str
    esquerda: No
    direita:  No

@dataclass
class OperacaoUnaria(No):
    op:      str
    operando: No

@dataclass
class DeclaracaoLet(No):
    nome:  str
    valor: No
    linha: int

@dataclass
class Atribuicao(No):
    nome:  str
    valor: No
    linha: int

@dataclass
class ComandoPrint(No):
    valor: No
    linha: int

@dataclass
class Programa(No):
    comandos: list = field(default_factory=list)


# padrão visitor — despacha pro método certo automaticamente
class Visitor:
    def visitar(self, no):
        metodo = getattr(self, f'visitar_{type(no).__name__}', self.generico)
        return metodo(no)

    def generico(self, no):
        raise NotImplementedError(f'visitar_{type(no).__name__} não implementado')


class ImpressorAST(Visitor):
    def __init__(self):
        self.nivel = 0

    def _print(self, texto):
        print('  ' * self.nivel + texto)

    def visitar_Programa(self, no):
        self._print('Programa')
        self.nivel += 1
        for cmd in no.comandos:
            self.visitar(cmd)
        self.nivel -= 1

    def visitar_DeclaracaoLet(self, no):
        self._print(f'let {no.nome} =')
        self.nivel += 1
        self.visitar(no.valor)
        self.nivel -= 1

    def visitar_Atribuicao(self, no):
        self._print(f'{no.nome} =')
        self.nivel += 1
        self.visitar(no.valor)
        self.nivel -= 1

    def visitar_ComandoPrint(self, no):
        self._print('print')
        self.nivel += 1
        self.visitar(no.valor)
        self.nivel -= 1

    def visitar_OperacaoBinaria(self, no):
        self._print(f'op: {no.op}')
        self.nivel += 1
        self.visitar(no.esquerda)
        self.visitar(no.direita)
        self.nivel -= 1

    def visitar_OperacaoUnaria(self, no):
        self._print(f'unario: {no.op}')
        self.nivel += 1
        self.visitar(no.operando)
        self.nivel -= 1

    def visitar_LiteralInt(self, no):
        self._print(f'int: {no.valor}')

    def visitar_LiteralFloat(self, no):
        self._print(f'float: {no.valor}')

    def visitar_Variavel(self, no):
        self._print(f'var: {no.nome}')

print('Nós da AST carregados')

Nós da AST carregados


In [39]:
# ============================================================
# FASE 2 — PARSER
# ============================================================

## Fase 2 — Análise Sintática (Parser)

O Parser pega a lista de tokens e constrói a AST. Ele verifica se o código segue a gramática da linguagem — não apenas se as palavras existem, mas se estão na ordem certa.

Se o Lexer é como verificar se as palavras existem num dicionário, o Parser verifica se a frase faz sentido gramaticalmente.

### `ErroSintatico`
Lançado quando a estrutura está errada — por exemplo, `let x = 2 +` (expressão incompleta) ou `print(10 + 5` (parêntese não fechado).

### Classe `Parser`
Os métodos `parse_expr()`, `parse_termo()`, `parse_unario()` e `parse_primario()` formam uma cadeia que implementa a **precedência de operadores**:

- `parse_expr` cuida de `+` e `-` (menor prioridade)
- `parse_termo` cuida de `*` e `/` (maior prioridade)
- `parse_unario` cuida da negação `-x`
- `parse_primario` lida com valores diretos: números, variáveis e `(...)`

Por que isso garante a precedência? Porque `*` e `/` são resolvidos num nível mais profundo da recursão, antes de `+` e `-`. Por isso `2 + 3 * 4` resulta em `14` e não `20`.

```
parse_expr()       →  trata + e -  (menor prioridade)
  └── parse_termo()    →  trata * e /  (maior prioridade)
        └── parse_unario()   →  trata -x
              └── parse_primario()  →  números, variáveis, (...)
```

In [40]:
class ErroSintatico(Exception):
    def __init__(self, msg, linha):
        super().__init__(f'Erro Sintático (linha {linha}): {msg}')
        self.linha = linha


class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos    = 0

    def atual(self):
        return self.tokens[self.pos]

    def tipo_atual(self):
        return self.tokens[self.pos].tipo

    def avancar(self):
        tok = self.tokens[self.pos]
        if self.pos < len(self.tokens) - 1:
            self.pos += 1
        return tok

    def esperar(self, tipo):
        tok = self.atual()
        if tok.tipo != tipo:
            raise ErroSintatico(
                f"esperava '{tipo.name}', encontrei '{tok.valor}'", tok.linha
            )
        return self.avancar()

    def pular_newlines(self):
        while self.tipo_atual() == TokenType.NEWLINE:
            self.avancar()

    def bater(self, *tipos):
        return self.tipo_atual() in tipos

    def parse(self):
        arvore = Programa()
        self.pular_newlines()
        while not self.bater(TokenType.EOF):
            cmd = self.parse_comando()
            if cmd is not None:
                arvore.comandos.append(cmd)
            self.pular_newlines()
        return arvore

    def parse_comando(self):
        tok = self.atual()

        if tok.tipo == TokenType.LET:
            return self.parse_let()
        elif tok.tipo == TokenType.PRINT:
            return self.parse_print()
        elif tok.tipo == TokenType.IDENTIFIER:
            # verifica se é atribuição (x = ...) ou só uma expressão
            proximo = self.tokens[self.pos + 1] if self.pos + 1 < len(self.tokens) else None
            if proximo and proximo.tipo == TokenType.ASSIGN:
                return self.parse_atribuicao()
            return self.parse_expr()
        elif tok.tipo in (TokenType.NEWLINE, TokenType.SEMICOLON):
            self.avancar()
            return None
        else:
            raise ErroSintatico(f"não esperava '{tok.valor}' aqui", tok.linha)

    def parse_let(self):
        tok_let = self.avancar()
        nome    = self.esperar(TokenType.IDENTIFIER)
        self.esperar(TokenType.ASSIGN)
        valor = self.parse_expr()
        return DeclaracaoLet(nome.valor, valor, tok_let.linha)

    def parse_atribuicao(self):
        nome = self.avancar()
        self.avancar()  # pula o '='
        valor = self.parse_expr()
        return Atribuicao(nome.valor, valor, nome.linha)

    def parse_print(self):
        tok = self.avancar()
        self.esperar(TokenType.LPAREN)
        valor = self.parse_expr()
        self.esperar(TokenType.RPAREN)
        return ComandoPrint(valor, tok.linha)

    # cuida de + e - (menor prioridade)
    def parse_expr(self):
        esq = self.parse_termo()
        while self.bater(TokenType.PLUS, TokenType.MINUS):
            op  = self.avancar()
            dir = self.parse_termo()
            esq = OperacaoBinaria(op.valor, esq, dir)
        return esq

    # cuida de * e / (maior prioridade)
    def parse_termo(self):
        esq = self.parse_unario()
        while self.bater(TokenType.STAR, TokenType.SLASH):
            op  = self.avancar()
            dir = self.parse_unario()
            esq = OperacaoBinaria(op.valor, esq, dir)
        return esq

    def parse_unario(self):
        if self.bater(TokenType.MINUS):
            op = self.avancar()
            return OperacaoUnaria(op.valor, self.parse_unario())
        return self.parse_primario()

    def parse_primario(self):
        tok = self.atual()

        if tok.tipo == TokenType.INTEGER:
            self.avancar()
            return LiteralInt(tok.valor)
        elif tok.tipo == TokenType.FLOAT:
            self.avancar()
            return LiteralFloat(tok.valor)
        elif tok.tipo == TokenType.IDENTIFIER:
            self.avancar()
            return Variavel(tok.valor, tok.linha)
        elif tok.tipo == TokenType.LPAREN:
            self.avancar()
            expr = self.parse_expr()
            self.esperar(TokenType.RPAREN)
            return expr
        elif tok.tipo == TokenType.EOF:
            raise ErroSintatico('fim de arquivo inesperado', tok.linha)
        else:
            raise ErroSintatico(f"não esperava '{tok.valor}' aqui", tok.linha)

print('Parser carregado')

Parser carregado


In [41]:
# ============================================================
# FASE 3 — ANÁLISE SEMÂNTICA
# ============================================================

## Fase 3 — Análise Semântica

Mesmo que o código seja sintaticamente correto, ele pode não fazer sentido. Essa fase verifica o **significado** do programa.

Exemplo clássico: `print(x)` é sintaticamente válido, mas semanticamente errado se `x` nunca foi declarado com `let`.

### `ErroSemantico`
Lançado para coisas como: usar variável sem declarar, declarar a mesma variável duas vezes, ou tentar reatribuir sem ter declarado antes.

### `TabelaDeSimbolos`
É basicamente um dicionário que guarda o nome e o tipo de cada variável declarada. Os três métodos fazem o seguinte: `declarar` registra uma variável nova (e reclama se já existe), `atualizar` muda o tipo de uma existente (e reclama se não foi declarada), e `buscar` consulta o tipo de uma variável (e reclama se não existe).

### `AnalisadorSemantico`
Percorre a AST verificando as regras e **inferindo os tipos** das expressões. A regra de tipos é simples: inteiro + inteiro = inteiro, qualquer coisa com float = float, e divisão **sempre** retorna float — mesmo `10 / 2` dá `5.0`.

In [42]:
class ErroSemantico(Exception):
    def __init__(self, msg, linha):
        super().__init__(f'Erro Semântico (linha {linha}): {msg}')
        self.linha = linha


class TabelaDeSimbolos:
    def __init__(self):
        self.variaveis = {}

    def declarar(self, nome, tipo, linha):
        if nome in self.variaveis:
            raise ErroSemantico(f"variável '{nome}' já foi declarada", linha)
        self.variaveis[nome] = tipo

    def atualizar(self, nome, tipo, linha):
        if nome not in self.variaveis:
            raise ErroSemantico(f"use 'let {nome} = ...' antes de atribuir", linha)
        self.variaveis[nome] = tipo

    def buscar(self, nome, linha):
        if nome not in self.variaveis:
            raise ErroSemantico(f"variável '{nome}' usada antes de ser declarada", linha)
        return self.variaveis[nome]


class AnalisadorSemantico(Visitor):
    def __init__(self):
        self.tabela = TabelaDeSimbolos()

    def analisar(self, programa):
        self.visitar(programa)
        return self.tabela

    def visitar_Programa(self, no):
        for cmd in no.comandos:
            self.visitar(cmd)

    def visitar_DeclaracaoLet(self, no):
        tipo = self.visitar(no.valor)
        self.tabela.declarar(no.nome, tipo, no.linha)

    def visitar_Atribuicao(self, no):
        tipo = self.visitar(no.valor)
        self.tabela.atualizar(no.nome, tipo, no.linha)

    def visitar_ComandoPrint(self, no):
        self.visitar(no.valor)

    def visitar_OperacaoBinaria(self, no):
        tipo_esq = self.visitar(no.esquerda)
        tipo_dir = self.visitar(no.direita)
        # divisão sempre vira float
        if no.op == '/':
            return 'float'
        # se um dos lados for float, o resultado é float
        if 'float' in (tipo_esq, tipo_dir):
            return 'float'
        return 'int'

    def visitar_OperacaoUnaria(self, no):
        return self.visitar(no.operando)

    def visitar_LiteralInt(self, no):
        return 'int'

    def visitar_LiteralFloat(self, no):
        return 'float'

    def visitar_Variavel(self, no):
        return self.tabela.buscar(no.nome, no.linha)

print('Analisador semântico carregado')

Analisador semântico carregado


## Fases 4 e 5 — Geração de Código e Máquina Virtual

### `GeradorDeCodigo`
Percorre a AST e gera **bytecode** — uma lista de instruções simples para uma máquina virtual de pilha. É uma linguagem intermediária: mais baixo nível que o código original, mas não é código de máquina real.

As instruções geradas são: `EMPURRAR` (coloca valor na pilha), `CARREGAR` (carrega variável da memória), `GUARDAR` (salva na memória), `SOMAR`, `SUBTRAIR`, `MULTIPLICAR`, `DIVIDIR` (operações aritméticas), `NEGAR` (inverte o sinal), `IMPRIMIR` (imprime o topo da pilha), `HALT` (para a execução).

### `MaquinaVirtual`
Executa o bytecode. Ela tem uma pilha (`pilha`), uma memória de variáveis (`memoria`) e uma lista de saída (`saida`).

A lógica de pilha é simples: para calcular `2 + 3 * 4`, o bytecode empurra os valores e vai operando:

EMPURRAR 2   →  pilha: [2]

EMPURRAR 3   →  pilha: [2, 3]

EMPURRAR 4   →  pilha: [2, 3, 4]

MULTIPLICAR  →  pilha: [2, 12]

SOMAR        →  pilha: [14]

A VM também detecta divisão por zero e lança `RuntimeError`.

### `compilar_e_rodar`
Função que junta tudo: chama o `Lexer`, depois o `Parser`, depois o `AnalisadorSemantico`, depois o `GeradorDeCodigo` e por fim a `MaquinaVirtual`. Com `verbose=True` mostra os tokens, a AST e o bytecode gerado — ótimo pra entender cada etapa.

In [43]:
# ============================================================
# FASES 4+5 — GERAÇÃO DE CÓDIGO + VM
# ============================================================

In [44]:
class GeradorDeCodigo(Visitor):
    def __init__(self):
        self.bytecode = []

    def emitir(self, *instrucao):
        self.bytecode.append(instrucao)

    def gerar(self, programa):
        self.visitar(programa)
        self.emitir('HALT')
        return self.bytecode

    def visitar_Programa(self, no):
        for cmd in no.comandos:
            self.visitar(cmd)

    def visitar_DeclaracaoLet(self, no):
        self.visitar(no.valor)
        self.emitir('GUARDAR', no.nome)

    def visitar_Atribuicao(self, no):
        self.visitar(no.valor)
        self.emitir('GUARDAR', no.nome)

    def visitar_ComandoPrint(self, no):
        self.visitar(no.valor)
        self.emitir('IMPRIMIR')

    def visitar_OperacaoBinaria(self, no):
        self.visitar(no.esquerda)
        self.visitar(no.direita)
        tabela_ops = {'+': 'SOMAR', '-': 'SUBTRAIR', '*': 'MULTIPLICAR', '/': 'DIVIDIR'}
        self.emitir(tabela_ops[no.op])

    def visitar_OperacaoUnaria(self, no):
        self.visitar(no.operando)
        self.emitir('NEGAR')

    def visitar_LiteralInt(self, no):
        self.emitir('EMPURRAR', no.valor)

    def visitar_LiteralFloat(self, no):
        self.emitir('EMPURRAR', no.valor)

    def visitar_Variavel(self, no):
        self.emitir('CARREGAR', no.nome)


class MaquinaVirtual:
    def __init__(self):
        self.pilha   = []
        self.memoria = {}
        self.saida   = []

    def rodar(self, bytecode):
        for instrucao in bytecode:
            op   = instrucao[0]
            args = instrucao[1:]

            if op == 'EMPURRAR':
                self.pilha.append(args[0])

            elif op == 'CARREGAR':
                self.pilha.append(self.memoria[args[0]])

            elif op == 'GUARDAR':
                self.memoria[args[0]] = self.pilha.pop()

            elif op == 'SOMAR':
                b = self.pilha.pop()
                a = self.pilha.pop()
                self.pilha.append(a + b)

            elif op == 'SUBTRAIR':
                b = self.pilha.pop()
                a = self.pilha.pop()
                self.pilha.append(a - b)

            elif op == 'MULTIPLICAR':
                b = self.pilha.pop()
                a = self.pilha.pop()
                self.pilha.append(a * b)

            elif op == 'DIVIDIR':
                b = self.pilha.pop()
                a = self.pilha.pop()
                if b == 0:
                    raise RuntimeError('divisão por zero')
                self.pilha.append(a / b)

            elif op == 'NEGAR':
                self.pilha.append(-self.pilha.pop())

            elif op == 'IMPRIMIR':
                valor = self.pilha.pop()
                texto = str(valor)
                self.saida.append(texto)
                print(texto)

            elif op == 'HALT':
                break

        return self.saida


def compilar_e_rodar(codigo, verbose=False):
    # fase 1 — lexer
    tokens = Lexer(codigo).tokenizar()
    if verbose:
        print('=== TOKENS ===')
        for t in tokens:
            print(' ', t)

    # fase 2 — parser
    arvore = Parser(tokens).parse()
    if verbose:
        print('\n=== AST ===')
        ImpressorAST().visitar(arvore)

    # fase 3 — semântica
    AnalisadorSemantico().analisar(arvore)

    # fase 4 — geração de código
    bytecode = GeradorDeCodigo().gerar(arvore)
    if verbose:
        print('\n=== BYTECODE ===')
        for i in bytecode:
            print(' ', i)

    # fase 5 — execução
    return MaquinaVirtual().rodar(bytecode)

print('Compilador pronto!')

Compilador pronto!


### 2.4 — Tratamento de erros

O compilador detecta três tipos de erro em fases diferentes:

- **`ErroLexico`** (Fase 1): caractere desconhecido como `@`
- **`ErroSemantico`** (Fase 3): variável usada sem ter sido declarada com `let`
- **`RuntimeError`** (Fase 5): divisão por zero detectada pela VM em tempo de execução

In [45]:
# Erro Léxico: caractere inválido
try:
    compilar_e_rodar("let x = 10 @ 5")
except ErroLexico as e:
    print(e)

Erro Léxico (linha 1, col 12): caractere '@' não reconhecido


In [46]:
# Erro Semântico: variável não declarada
try:
    compilar_e_rodar("print(y)")
except ErroSemantico as e:
    print(e)

Erro Semântico (linha 1): variável 'y' usada antes de ser declarada


In [47]:
# Erro de Execução: divisão por zero
try:
    compilar_e_rodar("print(10 / 0)")
except RuntimeError as e:
    print(e)

divisão por zero


## Testes Automatizados

Roda 24 casos de teste pra validar o compilador. Os 16 primeiros verificam se os resultados estão corretos (adição, subtração, precedência, variáveis, floats, comentários, etc.). Os 8 últimos verificam se os erros certos são lançados nas situações erradas — um teste de erro só passa se a exceção esperada for lançada.

In [48]:
import traceback

testes = [
    ('adicao',              lambda: compilar_e_rodar('let a=3\nlet b=4\nprint(a+b)') == ['7']),
    ('subtracao',           lambda: compilar_e_rodar('let a=10\nlet b=3\nprint(a-b)') == ['7']),
    ('multiplicacao',       lambda: compilar_e_rodar('let a=6\nlet b=7\nprint(a*b)') == ['42']),
    ('divisao float',       lambda: compilar_e_rodar('print(10/4)') == ['2.5']),
    ('divisao sempre float',lambda: compilar_e_rodar('print(10/2)') == ['5.0']),
    ('precedencia *>+',     lambda: compilar_e_rodar('print(2+3*4)') == ['14']),
    ('parenteses',          lambda: compilar_e_rodar('print((2+3)*4)') == ['20']),
    ('encadeado',           lambda: compilar_e_rodar('print(10-3-2)') == ['5']),
    ('parenteses aninhados',lambda: compilar_e_rodar('print((2+(3*4))-1)') == ['13']),
    ('let basico',          lambda: compilar_e_rodar('let x=42\nprint(x)') == ['42']),
    ('atribuicao',          lambda: compilar_e_rodar('let x=10\nx=x+5\nprint(x)') == ['15']),
    ('float',               lambda: compilar_e_rodar('let pi=3.14\nprint(pi)') == ['3.14']),
    ('multiplos prints',    lambda: compilar_e_rodar('let a=10\nlet b=3\nprint(a+b)\nprint(a-b)') == ['13','7']),
    ('int+float=float',     lambda: compilar_e_rodar('let x=1+0.5\nprint(x)') == ['1.5']),
    ('negacao unaria',      lambda: compilar_e_rodar('let x=5\nprint(-x+10)') == ['5']),
    ('comentario',          lambda: compilar_e_rodar('let x=10 # ok\nprint(x)') == ['10']),
]

erros = [
    ('erro lexico @',       lambda: (Lexer('let x=10@5').tokenizar(), False),         ErroLexico),
    ('erro lexico $',       lambda: (Lexer('$10').tokenizar(), False),                ErroLexico),
    ('erro sintatico paren',lambda: (Parser(Lexer('print(10+5').tokenizar()).parse(), False),   ErroSintatico),
    ('erro sintatico expr', lambda: (Parser(Lexer('let x=2+').tokenizar()).parse(), False),     ErroSintatico),
    ('erro sem nao decl',   lambda: (AnalisadorSemantico().analisar(Parser(Lexer('print(y)').tokenizar()).parse()), False),               ErroSemantico),
    ('erro sem redecl',     lambda: (AnalisadorSemantico().analisar(Parser(Lexer('let x=1\nlet x=2').tokenizar()).parse()), False),       ErroSemantico),
    ('erro sem sem let',    lambda: (AnalisadorSemantico().analisar(Parser(Lexer('z=10').tokenizar()).parse()), False),                   ErroSemantico),
    ('divisao por zero',    lambda: (compilar_e_rodar('print(1/0)'), False),          RuntimeError),
]

ok = 0
falhou = 0

for nome, fn in testes:
    try:
        assert fn()
        print(f'  ✅ {nome}')
        ok += 1
    except Exception as e:
        print(f'  ❌ {nome}: {e}')
        falhou += 1

for nome, fn, tipo_erro in erros:
    try:
        fn()
        print(f'  ❌ {nome}: deveria ter lançado {tipo_erro.__name__}')
        falhou += 1
    except tipo_erro:
        print(f'  ✅ {nome}')
        ok += 1
    except Exception as e:
        print(f'  ❌ {nome}: erro inesperado — {e}')
        falhou += 1

print(f'\nResultado: {ok}/{ok + falhou} testes passaram')

7
  ✅ adicao
7
  ✅ subtracao
42
  ✅ multiplicacao
2.5
  ✅ divisao float
5.0
  ✅ divisao sempre float
14
  ✅ precedencia *>+
20
  ✅ parenteses
5
  ✅ encadeado
13
  ✅ parenteses aninhados
42
  ✅ let basico
15
  ✅ atribuicao
3.14
  ✅ float
13
7
  ✅ multiplos prints
1.5
  ✅ int+float=float
5
  ✅ negacao unaria
10
  ✅ comentario
  ✅ erro lexico @
  ✅ erro lexico $
  ✅ erro sintatico paren
  ✅ erro sintatico expr
  ✅ erro sem nao decl
  ✅ erro sem redecl
  ✅ erro sem sem let
  ✅ divisao por zero

Resultado: 24/24 testes passaram
